In [0]:
%pip install {spark.conf.get("conf.lib_path") + "/.internal/*.whl"}

## Permissions
Minimum permissions to run this are:
* Monitoring Metrics Publisher on DCR to write
* Monitoring Reader on DCR
* Monitoring Reader on DCE

# Setup

In [ ]:
from databricks.sdk.runtime import dbutils, spark
import dlt

In [0]:
import json
from dataclasses import dataclass

from azure.identity import ClientSecretCredential

from pyspark.sql import functions as F

from sentinel_helpers.log_analytics import get_dce, get_dcr
from sentinel_helpers.spark import AzureMonitorDataSource

spark.dataSource.register(AzureMonitorDataSource)

# Variables

In [0]:
audit_log_table_name = spark.conf.get("conf.audit_log_table_name")
outbound_network_table_name = spark.conf.get("conf.outbound_network_table_name")
system_table_names_map = {
    audit_log_table_name: "system.access.audit",
    outbound_network_table_name: "system.access.outbound_network",
}

starting_datetime = spark.conf.get("conf.starting_datetime")
include_workspace_ids = json.loads(spark.conf.get("conf.include_workspace_ids"))
exclude_workspace_ids = json.loads(spark.conf.get("conf.exclude_workspace_ids"))
processing_time = spark.conf.get("conf.processing_time")

tenant_id = spark.conf.get("conf.tenant_id")
subscription_id = spark.conf.get("conf.subscription_id")
sp_client_id = spark.conf.get("conf.sp_client_id")
sp_client_secret_scope = spark.conf.get("conf.sp_client_secret_scope")
sp_client_secret_key = spark.conf.get("conf.sp_client_secret_key")
resource_group_name = spark.conf.get("conf.resource_group_name")

sp_secret = dbutils.secrets.get(sp_client_secret_scope, sp_client_secret_key)

# Initialization

In [ ]:
@dataclass
class SystemTableFlowDetails:
    log_analytics_table_name: str
    system_table_name: str
    dcr_id: str
    dce_url: str
    raw_stream_declaration_name: str

In [0]:
flow_details: list[SystemTableFlowDetails] = list()

for log_analytics_table_name, system_table_name in system_table_names_map.items():

    data_collection_endpoint_name = f"{log_analytics_table_name}-dce"
    data_collection_rule_name = f"{log_analytics_table_name}-dcr"
    raw_stream_declaration_name = f"Custom-{log_analytics_table_name}RawData"

    credentials = ClientSecretCredential(
        tenant_id=tenant_id,
        client_id=sp_client_id,
        client_secret=sp_secret
    )

    dcr_id = get_dcr(credentials, subscription_id, resource_group_name, data_collection_rule_name).immutable_id
    dce_url = get_dce(credentials, subscription_id, resource_group_name, data_collection_endpoint_name).logs_ingestion.endpoint

    flow_details.append(
        SystemTableFlowDetails(
            log_analytics_table_name=log_analytics_table_name, 
            system_table_name=system_table_name, 
            dcr_id=dcr_id, 
            dce_url=dce_url, 
            raw_stream_declaration_name=raw_stream_declaration_name
        )
    )

## Core

In [ ]:
for flow in flow_details:
    azure_monitor_options = {
        "dce_url": flow.dce_url,
        "dcr_id": flow.dcr_id,
        "dcs": flow.raw_stream_declaration_name,
        "tenant_id": tenant_id,
        "client_id": sp_client_id,
        "client_secret": sp_secret,
        "body_col": "row_json",
    }

    if processing_time:
        spark_conf = {
            "pipelines.trigger.interval": processing_time
        }
    else:
        spark_conf = None

    sink_name = f"{flow.log_analytics_table_name}_sink"

    dlt.create_sink(sink_name, "azure-monitor", azure_monitor_options)

    @dlt.append_flow(
        target=sink_name, 
        name=f"{flow.log_analytics_table_name}_flow",
        spark_conf=spark_conf,
    )
    def log_analytics_table_flow(table_name=flow.system_table_name):
        input_df = (
            spark.readStream
            .option("skipChangeCommits", "true")
            .table(table_name)
            .filter(F.col("event_time") >= starting_datetime)
        )
        
        if include_workspace_ids:
            input_df = input_df.filter(F.col("workspace_id").isin(include_workspace_ids))
        if exclude_workspace_ids:
            input_df = input_df.filter(~F.col("workspace_id").isin(exclude_workspace_ids))        

        return (
            input_df.withColumn("TimeGenerated", F.col("event_time"))
                .select(F.to_json(F.struct("*")).alias("row_json"))
        )